# Detekcja tablic: trening, ewaluacja, wyrównanie, wycinki dla OCR

Cała część detekcyjna potoku w jednym miejscu. Kroki zapisują wyniki w `runs/` i `results/`,
więc każdy można uruchomić osobno (np. sama ewaluacja z gotowymi wagami).

**Gdzie uruchamiać**

* **Kaggle** (zalecane): nowy notebook, *File -> Import Notebook* z tego pliku, *Accelerator: GPU*,
  *Internet: on*, *Add Input*: `filipstanczyk/eu-license-plates-alpr`. Zbiór jest od razu podpięty.
* **Colab**: *Runtime -> GPU*. Zbiór (ok. 4.5 GB) pobiera się do `data/combined/` przez API Kaggle:
  dodaj w *Secrets* (ikona klucza) `KAGGLE_USERNAME` i `KAGGLE_KEY` z tokenu konta dodanego do zbioru.
* **Lokalnie**: `pip install -e .` i `python3 scripts/get_data.py`; trening na CPU tylko do testów
  (`EPOCHS = 1`, `FRACTION = 0.1`).

Na Kaggle i w Colab pierwsza komórka klonuje repozytorium i instaluje pakiet `alpr`.

In [ ]:
import contextlib
import os
import subprocess
import sys
from pathlib import Path

REPO, BRANCH = "https://github.com/FStanczyk/LicensePlatesN2Project.git", "main"
KAGGLE, COLAB = Path("/kaggle/working").exists(), "google.colab" in sys.modules

if KAGGLE or COLAB:
    root = Path("/kaggle/working" if KAGGLE else "/content") / "LicensePlatesN2Project"
    if root.exists():
        subprocess.run(["git", "-C", str(root), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    os.chdir(root)
    sys.path.insert(0, str(root))
if COLAB:
    from google.colab import userdata

    for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        with contextlib.suppress(Exception):
            os.environ[key] = userdata.get(key)

## Parametry

* `MODEL`, `IMGSZ`: wagi startowe i rozdzielczość; do porównania `yolov8n.pt`, `yolo11n.pt`, `yolo11s.pt` oraz 640 i 960 px.
* `NAME`: nazwa przebiegu, od niej zależą katalogi `runs/detect/<NAME>` i `results/*/<NAME>`.
* `WEIGHTS`: ścieżka do gotowego `best.pt`, wtedy trening jest pomijany. Trening jest też pomijany,
  gdy `runs/detect/<NAME>/weights/best.pt` już istnieje (`RETRAIN = True` wymusza nowy).
* `CONF`: próg pewności dla P, R, F1 i wycinków; `SPLIT`: podział do ewaluacji.

In [ ]:
MODEL, IMGSZ = "yolo11n.pt", 640
EPOCHS, BATCH, PATIENCE, FRACTION = 100, 16, 20, 1.0
NAME = f"{Path(MODEL).stem}_{IMGSZ}"
WEIGHTS, RETRAIN = None, False
CONF, SPLIT = 0.25, "test"

## Dane

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from alpr.config import COMBINED, RESULTS, RUNS, default_dataset
from alpr.dataset import load_plates, write_csv
from alpr.detection import PlateAligner, PlateDetector
from alpr.detection.metrics import boxes_by_image, evaluate_by_group, summarize_alignment
from alpr.detection.pipeline import DETECTION_FIELDS, crop_plates, evaluate_alignment, predict_boxes
from alpr.detection.training import train_detector
from alpr.download import fetch_kaggle, missing_files

DATASET = default_dataset()
if not (DATASET / "plates.csv").exists():
    fetch_kaggle(COMBINED)
    DATASET = COMBINED
    assert not missing_files(DATASET)[1], "niekompletny zbiór, uruchom komórkę ponownie"
print(f"Zbiór: {DATASET}")

plates = load_plates(DATASET)
full = plates[plates.sample_type == "full_photo"]
display(plates.groupby(["sample_type", "split"]).size().unstack(fill_value=0))
display(pd.crosstab([full.source, full.corrupted], full.exposure, margins=True))

Przykładowe zdjęcia z prawdziwymi ramkami (zielone) i narożnikami (czerwone), tylko oryginały.

In [ ]:
def show(images, titles, cols=4, size=4):
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 0.75 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, title in zip(axes.flat, images, titles, strict=False):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


def draw(image, boxes=(), color=(0, 255, 0), polygons=(), img=None):
    img = cv2.imread(str(DATASET / image)) if img is None else img
    t = max(2, img.shape[1] // 300)
    for x1, y1, x2, y2 in boxes:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, t)
    for poly in polygons:
        cv2.polylines(img, [np.array(poly.split(), float).reshape(4, 2).astype(np.int32)], True, (0, 0, 255), t)
    return img


sample = full[full.corrupted == "original"].groupby("exposure").sample(n=2, replace=True, random_state=0)
show(
    [
        draw(r.image, [(r.x1, r.y1, r.x2, r.y2)], polygons=[r.polygon] if isinstance(r.polygon, str) else [])
        for r in sample.itertuples()
    ],
    [f"{r.source}, {r.exposure}, {r.lighting}" for r in sample.itertuples()],
)

## Trening

Fine-tuning z wag COCO, stałe ziarno, wybór epoki na val. Na GPU (T4/P100) ok. 1 min na epokę.

In [ ]:
import torch

best = Path(WEIGHTS) if WEIGHTS else RUNS / "detect" / NAME / "weights" / "best.pt"
if RETRAIN or not best.exists():
    best = train_detector(
        DATASET / "detection" / "data.yaml",
        MODEL,
        RUNS / "detect",
        NAME,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        fraction=FRACTION,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=4,
    )
print(f"Wagi: {best}")
if (best.parent.parent / "results.png").exists():
    display(Image(filename=str(best.parent.parent / "results.png"), width=900))

## Ewaluacja detekcji

P, R, F1 przy progu `CONF`, AP50 i AP50-95 (predykcje z progiem 0.001) dla całego podziału i w grupach.
Zdjęcie z tablicami o różnych atrybutach trafia do grupy `mixed`.

In [ ]:
gt = load_plates(DATASET, sample_type="full_photo", split=SPLIT)
det_out = RESULTS / "detection_eval" / NAME
det_out.mkdir(parents=True, exist_ok=True)
preds = predict_boxes(PlateDetector(best, 0.001, IMGSZ), sorted(gt.image.unique()), DATASET)
preds.to_csv(det_out / "predictions.csv", index=False)

metrics, plate_hits = evaluate_by_group(gt, preds, CONF)
metrics.to_csv(det_out / "metrics.csv", index=False, float_format="%.4f")
plate_hits.to_csv(det_out / "plates.csv", index=False)
with pd.option_context("display.max_rows", 100):
    display(metrics.round(3))

Recall i AP50 w grupach (pionowa kreska: wynik dla całego podziału).

In [ ]:
PANELS = ["exposure", "lighting", "corrupted", "source"]
SERIES = {"R": "#2a78d6", "AP50": "#eb6834"}

fig, axes = plt.subplots(1, len(PANELS), figsize=(4 * len(PANELS), 3.2), sharex=True)
overall = metrics.iloc[0]
for ax, group in zip(axes, PANELS, strict=True):
    part = metrics[metrics.group == group].sort_values("value")
    y = np.arange(len(part))
    for k, (col, color) in enumerate(SERIES.items()):
        ax.barh(y + (k - 0.5) * 0.38, part[col], 0.34, color=color, label=col)
        ax.axvline(overall[col], color=color, lw=1, ls=":")
    ax.set_yticks(y, [f"{v} ({n})" for v, n in zip(part.value, part.images, strict=True)], fontsize=9)
    ax.set_ylim(max(len(part), 3) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.set_title(group, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

Przeoczone tablice (zielone: prawda, czerwone: predykcje z pewnością >= `CONF`).

In [ ]:
pred_boxes = boxes_by_image(preds, CONF)
missed = gt[gt.plate_id.isin(plate_hits.plate_id[plate_hits.detected == 0])]
print(f"Przeoczone: {len(missed)} z {len(gt)} tablic")
missed_sample = missed.sample(n=min(8, len(missed)), random_state=0)
if len(missed_sample):
    show(
        [
            draw(r.image, pred_boxes.get(r.image, []), (0, 0, 255), img=draw(r.image, [(r.x1, r.y1, r.x2, r.y2)]))
            for r in missed_sample.itertuples()
        ],
        [f"{r.plate_id}: {r.exposure}, {r.artificial_corruption}" for r in missed_sample.itertuples()],
    )

## Wyrównanie perspektywy

Zwykły wycinek ramki (`box`) a `PlateAligner` (`aligned`) względem prawdziwych narożników, na ramkach
z detektora. Narożniki z perspektywą ma tylko data2 (UC3M-LP), w data1 są prostokąty.
`better`: odsetek tablic, na których wyrównanie poprawia IoU o ponad 0.01.

In [ ]:
align_out = RESULTS / "alignment_eval" / NAME
(align_out / "examples").mkdir(parents=True, exist_ok=True)
res = evaluate_alignment(gt[gt.polygon.notna()], DATASET, PlateAligner(), pred_boxes, align_out / "examples", 12)
res.to_csv(align_out / "plates.csv", index=False, float_format="%.4f")
summary = summarize_alignment(res)
summary.reset_index().to_csv(align_out / "summary.csv", index=False, float_format="%.4f")
display(summary.round(3))

Przykłady: od góry wycinek ramki, wyrównany, z prawdziwych narożników.

In [ ]:
examples = sorted((align_out / "examples").glob("*.png"))
show([cv2.imread(str(p)) for p in examples], [p.stem for p in examples], cols=6, size=2.5)

## Wycinki dla OCR

Wyprostowane wycinki z podziału `SPLIT` i `detections.csv` (wejście dla Golby). Zdjęcie bez detekcji
ma jeden wiersz z pustym `det_id`.

In [ ]:
crops_out = RESULTS / "detections" / NAME
images = sorted(gt.image.unique())
rows = crop_plates(PlateDetector(best, CONF, IMGSZ), images, DATASET, crops_out, PlateAligner())
write_csv(crops_out / "detections.csv", rows, DETECTION_FIELDS)
detections = pd.DataFrame(rows, columns=DETECTION_FIELDS)
print(f"{len(images)} zdjęć, {detections.det_id.notna().sum()} tablic, {detections.det_id.isna().sum()} bez detekcji")
display(detections["align"].value_counts())

shown = detections.dropna(subset=["crop"]).sample(n=min(12, detections.crop.notna().sum()), random_state=0)
show(
    [cv2.imread(str(crops_out / c)) for c in shown.crop],
    [f"{a}, {c:.2f}" for a, c in zip(shown["align"], shown.conf, strict=True)],
    cols=6,
    size=2.5,
)

## Spakowanie wyników

Na Kaggle zip trafia do *Output* (`/kaggle/working`), w Colab pobiera się automatycznie.
Lokalnie wyniki są już w `runs/` i `results/`.

In [ ]:
import shutil
import tempfile

if KAGGLE or COLAB:
    with tempfile.TemporaryDirectory() as tmp:
        pack = Path(tmp) / NAME
        shutil.copytree(best.parent.parent, pack / "run", ignore=shutil.ignore_patterns("last.pt"))
        for sub in ("detection_eval", "alignment_eval", "detections"):
            shutil.copytree(RESULTS / sub / NAME, pack / sub)
        archive = shutil.make_archive(str(Path("/kaggle/working" if KAGGLE else "/content") / NAME), "zip", pack)
    print(f"Archiwum: {archive}")
    if COLAB:
        from google.colab import files

        files.download(archive)